# Wordle 任务介绍

在深入 SFT 概念之前，我们先来理解 Wordle 这个游戏——它是贯穿本教程和后续 RL 教程的核心任务。

---

## 游戏规则

Wordle 是一个经典的英文猜词游戏，规则简单但富有策略深度：

- **目标**：猜出一个隐藏的 5 字母英文单词。
- **回合限制**：最多猜 6 轮。
- **反馈机制**：每轮猜词后，系统对每个字母位置给出三种反馈：
  - **G（Green，绿色）**：字母正确，位置正确。
  - **Y（Yellow，黄色）**：字母正确，但位置不对。
  - **X（错误）**：不属于这个词的字母。特殊情况：重复字母因此可能有一个是 G/Y、另一个是 X，因为在先匹配 G、再按剩余字母数量匹配 Y 后，剩余的这个字母也属于错误。例如目标词 `APPLE`、猜词 `ALLEY` 的反馈是 `G Y X Y X`；目标词只有一个 `L`，所以猜词中的第二个 `L` 为 X。


```text
示例（目标词：SMILE）
第1轮：CRANE → C-r-a-n-e → X X X X G
        （E 的字母和位置都正确）
第2轮：SLIME → S-l-i-m-e → G Y G Y G
        （S、I、E 位置正确；L、M 字母正确但位置不对）
第3轮：SMILE → s-m-i-l-e → G G G G G
        （全部正确，游戏结束！）
```


## 为什么选择 Wordle 作为训练任务？

Wordle 非常适合作为两阶段训练（SFT + RL）的演示任务，主要基于以下考虑：

### 1. 天然的多轮决策属性

Wordle 并非孤立的单步问题，模型必须依据每一轮获得的反馈动态调整后续猜测，这使其天然具备多轮交互与序列决策的特性。

### 2. SFT 阶段的核心目标

训练数据中的每条回复均采用固定格式：`<think>...</think><guess>[word]</guess>`。SFT 的首要任务就是让模型学会严格遵循这一输出模板。在评测时，我们会利用 Prime-RL 的 verifier 判定是否能够解析 `<guess>` 标签中的单词，并与标准答案进行比对。同时，训练数据包含了完整的多轮对话样本，模型会观察到“上一轮猜测 + 收到的反馈 → 下一轮猜测”的典型模式，SFT 能够有效模仿并固化这种交互格式。

为验证 SFT 的效果，我们将在相同随机输入条件下，分别运行基座模型（未经训练）和 SFT 后的模型，通过对比两者的输出，直观检查格式规范性是否得到显著提升。

### 3. RL 阶段的后续方向

在 SFT 打好格式基础之后，后续的 RL 阶段将聚焦于策略优化、推理能力增强以及更复杂的决策分析，从而进一步提升任务表现。

## 数据示例

```json
{
  "prompt": [
    {"role": "system", "content": "Follow the Wordle game instructions and required format."},
    {"role": "user", "content": "A secret 5-letter word has been chosen. You have 6 attempts."}
  ],
  "completion": [
    {"role": "assistant", "content": "<think>Test common letters.</think><guess>[crane]</guess>"},
    {"role": "user", "content": "CRANE -> X X G G G. You have 5 guesses left."},
    {"role": "assistant", "content": "<think>Keep A, N, E fixed.</think><guess>[plane]</guess>"}
  ],
  "answer": "plane",
  "reward": 1.0,
  "task": "wordle"
}
```


我们用的是 willcb/V3-wordle 这个数据集里的多轮游戏轨迹。这些轨迹提供了行为监督，但课程并没有验证每一步都是最优猜测。SFT 能学到轨迹里的格式和“根据反馈调整”的模式，但学完并不保证它比标签数据里的策略更好：它没有遵守 Wordle 的约束的能力，要靠后续RL阶段学习策略。

---

## 小结

- Wordle 是一个 5 字母、6 轮、G/Y/X 反馈的猜词游戏。
- 先标 G，再按剩余字母数量标 Y，最后标 X。
- 本课程用 SFT 模仿回复协议和已有轨迹，用 RL 奖励继续优化策略。
- 当前协议示例是 `<think>...</think><guess>[word]</guess>`；`format_reward` 与 `correct_answer` 衡量的内容不同。
- 数据用 `prompt + completion` 分段存储，processor 将二者拼成训练消息。

下一节，我们将深入 SFT 的核心概念，理解它是如何让模型学会输出正确格式的。

## 练习

1. （单选题）Wordle 游戏中，反馈 G 表示什么含义？
    A. 字母不在目标单词中
    B. 字母正确且位置正确
    C. 字母正确但位置错误
    D. 字母未被评估

2. （单选题）本课程用哪个评测指标观察 `<guess>` XML 格式的遵从程度？
    A. correct_answer
    B. partial_answer
    C. length_bonus
    D. format_reward

3. （单选题）本课程中，SFT 对多轮 Wordle 轨迹的直接学习作用是什么？
    A. 模仿轨迹中“历史反馈 → 下一次猜测”的行为
    B. 保证得到全局最优猜词策略
    C. 替代后续的行为评测
    D. 改变 Wordle 的 G/Y/X 规则


In [ ]:
!cat ./answer/01.02_answer.txt